# Gemma 4 31B Tool-Calling Fine-Tune

Based on the [official Unsloth Gemma4_(31B)-Text notebook](https://github.com/unslothai/notebooks/blob/main/nb/Gemma4_(31B)-Text.ipynb).

Dataset: [mtita/gemma4-tool-calling-training](https://huggingface.co/datasets/mtita/gemma4-tool-calling-training) (32,893 examples)

**Run All** to start training. Set your HF token in the Save cell at the bottom.


### Installation


In [ ]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# Gemma 4 needs transformers >= 5.5.0
!pip install git+https://github.com/huggingface/transformers.git
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;


In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio


### Load Model


In [ ]:
# Diagnostic: verify install worked before loading model
import transformers
print(f"transformers: {transformers.__version__}")
print(f"location: {transformers.__file__}")

from transformers import AutoConfig
try:
    config = AutoConfig.from_pretrained("unsloth/gemma-4-31B-it")
    print(f"AutoConfig OK: model_type={config.model_type}")
except Exception as e:
    print(f"AutoConfig FAILED: {type(e).__name__}: {e}")

try:
    from transformers import Gemma4ForConditionalGeneration
    print("Gemma4 architecture: SUPPORTED")
except ImportError:
    print("ERROR: Gemma4 NOT supported - transformers too old! Need >= 5.5.0")
    print("Run: pip install --no-deps git+https://github.com/huggingface/transformers.git")
    print("Then RESTART KERNEL and re-run from this cell")


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


### Add LoRA Adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 32,           # Higher rank for tool-calling precision
    lora_alpha = 32,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)


### Data Prep

We use the `gemma-4` chat template. Gemma 4 renders conversations like:
```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)


In [ ]:
from datasets import load_dataset
dataset = load_dataset("mtita/gemma4-tool-calling-training",
                        data_files="combined_training.jsonl",
                        split="train")
print(f"Loaded {len(dataset)} training examples")
print(dataset[0]["messages"][:2])


Map roles for Gemma 4 (system → user prefix, tool → user prefix) and apply chat template.


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        formatted = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "")
            if role == "system":
                formatted.append({"role": "user", "content": f"[System Instructions]\n{content}"})
                formatted.append({"role": "assistant", "content": "Understood."})
            elif role == "user":
                formatted.append({"role": "user", "content": content})
            elif role == "assistant":
                formatted.append({"role": "assistant", "content": content})
            elif role == "tool":
                tool_name = msg.get("name", "tool")
                formatted.append({"role": "user", "content": f"[Tool Result: {tool_name}]\n{content}"})
        try:
            text = tokenizer.apply_chat_template(
                formatted, tokenize=False, add_generation_prompt=False
            ).removeprefix('<bos>')
            texts.append(text)
        except Exception:
            texts.append("")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Formatted: {len(dataset)} examples")
print(dataset[0]["text"][:500])


### Train the model


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)


Only train on assistant outputs, ignore user inputs:


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)


In [ ]:
# Verify masking - should see full conversation
tokenizer.decode(trainer.train_dataset[0]["input_ids"])


In [ ]:
# Verify masking - should see ONLY assistant responses (rest is spaces)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


### Train!


In [ ]:
trainer_stats = trainer.train()


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")


### Test Inference


In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": """You have these tools:\n"
"- read(path): Read a file\n"
"- edit(path, old_text, new_text): Replace text in a file\n"
"- bash(command): Run a shell command\n\n"
"Fix the typo in src/utils.py where 'retrun' should be 'return'."""}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 512,
    use_cache = False,
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)


### Save LoRA adapters

**Set your HF token below!**


In [ ]:
model.save_pretrained("gemma_4_lora")
tokenizer.save_pretrained("gemma_4_lora")

# Uncomment and set your token to push to HuggingFace:
model.push_to_hub("mtita/gemma4-31b-tool-calling-lora", token = "YOUR_HF_TOKEN")
tokenizer.push_to_hub("mtita/gemma4-31b-tool-calling-lora", token = "YOUR_HF_TOKEN")


### Export to GGUF

Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.


In [ ]:
model.save_pretrained_gguf(
    "gemma_4_finetune",
    tokenizer,
    quantization_method = "q4_k_m",
)


In [ ]:
# Push GGUF to HuggingFace
model.push_to_hub_gguf(
    "mtita/gemma4-31b-tool-calling-q4_k_m-GGUF",
    tokenizer,
    quantization_method = "q4_k_m",
    token = "YOUR_HF_TOKEN",
)
